In [ ]:
import pandas as pd
import glob

def create_pattern():

    """Patterns to search"""

    news = pd.read_excel(f'../../../../social_influencers_africa/data/newspapers/urls_newspapers_za_ke_validated.xlsx')
    news = news[news['valid'] == True]
    newspapers = list(news[~news['url_site'].isna()]['url_site'])
    newspapers = [url.split('://')[1] for url in newspapers]
    patternnews = '|'.join([f'(?i){url}' for url in newspapers])
    factcheckers = list(news[news.factchecker == 1]['url_site'])
    factcheckers = [url.split('://')[1] for url in factcheckers]
    patternfactchecks = '|'.join([f'(?i){url}' for url in factcheckers])

    return patternnews, patternfactchecks


def journalistic_content(df, identifier, link_column):

    """Create dummies depending if post if content related to newsarticles of factchecks"""

    patternnews, patternfactchecks = create_pattern()

    # Create dummy if shared newsarticles
    df01 = df[~df[link_column].isna()]
    df02 = df[df[link_column].isna()]
    df02['link_newsarticle'] = 0
    dfnews = df01[df01[link_column].str.contains(patternnews)]
    dfnews['link_newsarticle'] = 1
    df = pd.concat([dfnews, df02, df01], ignore_index=True).reset_index(drop=True)
    df['link_newsarticle'] = df['link_newsarticle'].fillna(0)
    df = df.drop_duplicates(subset=[identifier], keep='first')

    # Create dummy if shared fact checks
    df01 = df[~df[link_column].isna()]
    df02 = df[df[link_column].isna()]
    df02['link_factcheck'] = 0
    dffacts = df01[df01[link_column].str.contains(patternfactchecks)]
    dffacts['link_factcheck'] = 1
    df = pd.concat([dffacts, df02, df01], ignore_index=True).reset_index(drop=True)
    df['link_factcheck'] = df['link_factcheck'].fillna(0)
    df = df.drop_duplicates(subset=[identifier], keep='first')

    return df


def expand_column(df, column):

    df0 = df[~df[column].isna()]
    df0 = df0.reset_index(drop=True)
    dfi = pd.DataFrame(df0[column].tolist())
    dfi = dfi[0].apply(pd.Series)
    df0 = pd.concat([df0, dfi], axis=1)
    df0 = df0.reset_index(drop=True)
    df1 = df[df[column].isna()].reset_index(drop=True)
    df = pd.concat([df0, df1], ignore_index=True)
    df = df.reset_index(drop=True)

    return df





In [ ]:
country = 'SA'
batch = '_batch2'

dft = pd.read_parquet(f'../../data/03-experiment/{country}/baseline/01-preprocess/influencers/tweets{batch}.parquet')
dft = dft.join(pd.json_normalize(dft['entities.urls']))
dft = journalistic_content(dft, 'id', 'expanded_url')
dft.to_parquet(
        f'../../data/03-experiment/{country}/baseline/01-preprocess/influencers/tweets_na{batch}.parquet.gzip',
        compression='gzip'
    )

country = 'KE'

dft = pd.read_parquet(f'../../data/03-experiment/{country}/baseline/01-preprocess/influencers/tweets{batch}.parquet')
dft = dft.join(pd.json_normalize(dft['entities.urls']))
dft = journalistic_content(dft, 'id', 'expanded_url')
dft.to_parquet(
        f'../../data/03-experiment/{country}/baseline/01-preprocess/influencers/tweets_na{batch}.parquet.gzip',
        compression='gzip'
    )

def summ_column(colname, idcol):
        """Summmarize column"""

        descrip = ['min', 'mean', 'max', 'sum', 'std']
        engsum = df.groupby(idcol).agg({colname: descrip})
        engsum.columns = [
            f'{colname}_min',
            f'{colname}_mean',
            f'{colname}_max',
            f'{colname}_sum',
            f'{colname}_std']
        return(engsum)


In [125]:
        df = pd.read_parquet(
            f'../../data/03-experiment/{country}/baseline/01-preprocess/influencers/tweets_na_batch2.parquet.gzip'
            )
        cols = [col for col in df.columns if 'public_metrics' in col]
        idcol = 'author_id'
        df[idcol] = df[idcol].astype(str)
        colname = [idcol]*len(cols)
        engage = pd.concat(
            list(map(summ_column, cols, colname)),
            axis=1
            ).reset_index()
        engage['interactions_count_sum'] = engage[[
            col for col in engage.columns if 'sum' in col]].sum(axis=1)
        engage.columns = engage.columns.str.replace(r'public_metrics.', '')
        users = pd.read_parquet(
            f'../../data/01-characterize/influencers/{country}/confirmed_influencers_batch2.parquet'
            )
        rename_users = {'id': 'author_id'}
        users = users.rename(columns=rename_users)
        users['author_id'] = users['author_id'].astype(str)
        tup_stats = {
            'link_factcheck': 'sum',
            'link_newsarticle': 'sum',
            'id': 'count'
            }
        stats_news = df.groupby(idcol, as_index=False).agg(tup_stats)
        rename_stats = {
            'link_factcheck': 'n_tweets.fc',
            'link_newsarticle': 'n_tweets.na',
            'id': 'n_tweets'
        }
        stats_news = stats_news.rename(columns=rename_stats)
        users = stats_news.merge(users, on=idcol, how='left')
        users = users.merge(engage, on=idcol, how='left')
        #users = users[(users['n_tweets.fc'] != 0) | (users['n_tweets.na'] != 0)].reset_index(drop=True)
        #users.to_parquet(
          #  f'../../data/03-experiment/{country}/baseline/01-preprocess/influencers/accounts_sa.parquet'
           # )


<ipython-input-125-0d59a2445e9e>:14: FutureWarning: The default value of regex will change from True to False in a future version.
  engage.columns = engage.columns.str.replace(r'public_metrics.', '')


In [127]:
len(users)

51

In [92]:
coun = 'ke'

users_ke_1=pd.read_parquet(f'../../../../social_influencers_africa/data/twitter/1-candidates/possible_users/users_{coun}_filtered.parquet')

users_ke_2=pd.read_parquet(f'../../../../social_influencers_africa/data/twitter/1-candidates/possible_users/users_{coun}_filtered_additional.parquet')
users_ke = pd.concat([users_ke_1, users_ke_2], axis=0)

In [93]:
users_ke.drop_duplicates('id', inplace=True)

In [94]:
users_ke.rename(columns={'id':'author_id'}, inplace=True)

In [95]:
users_ke['author_id'] = users_ke['author_id'].astype(str)

In [96]:
data = pd.merge(users,users_ke,on='author_id',how='left')

In [103]:
import tweetple

from tweetple import TweetPle

# Bearer token accesible via Twitter Developer Academic Research Track
bearer_token='AAAAAAAAAAAAAAAAAAAAAFpgZAEAAAAAbJS59UWzipi32ixd7LHtXov9olo%3D7gxD8Afshgj4munMXHLU08jzRdTpsAh4RZqq7VBofq1wAvkx1T'

# List of handle ids
ids = list(data[data['username'].isnull()].author_id)


missing_info = TweetPle.TweepleStreamer(ids, bearer_token).user_lookup()

0it [00:00, ?it/s]/Users/joaquinbarrutia/miniforge3/envs/env_tf/lib/python3.9/site-packages/tweetple/TweetPle.py:61: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_stats = df_stats.append(
1it [00:01,  1.35s/it]


In [104]:
no_info = data[data['username'].isnull()]
full_info = data[~data['username'].isnull()]

no_info = no_info[['author_id', 'n_tweets.fc', 'n_tweets.na', 'n_tweets', 'handle',
       'impression_count_min', 'impression_count_mean', 'impression_count_max',
       'impression_count_sum', 'impression_count_std', 'like_count_min',
       'like_count_mean', 'like_count_max', 'like_count_sum', 'like_count_std',
       'quote_count_min', 'quote_count_mean', 'quote_count_max',
       'quote_count_sum', 'quote_count_std', 'reply_count_min',
       'reply_count_mean', 'reply_count_max', 'reply_count_sum',
       'reply_count_std', 'retweet_count_min', 'retweet_count_mean',
       'retweet_count_max', 'retweet_count_sum', 'retweet_count_std',
       'interactions_count_sum']]

In [105]:
missing_info = missing_info[['id','location', 'created_at', 'description',
       'name', 'url', 'username', 'verified', 'public_metrics.followers_count',
       'public_metrics.following_count', 
                 'public_metrics.listed_count', 'public_metrics.tweet_count']].rename(columns=
                                                                                      {'id':'author_id',
                                                                                      'public_metrics.followers_count':'followers_count',
                                                                                      'public_metrics.listed_count':'listed_count',
                                                                                      'public_metrics.tweet_count':'tweet_count',
                                                                                      'public_metrics.following_count':'following_count'})

In [106]:
no_info = pd.merge(no_info,missing_info,on='author_id',how='left')

In [109]:
len(data)

58

In [108]:
data = pd.concat([full_info, no_info], axis=0).reset_index(drop=True)

In [111]:
from datetime import date
import datetime as dt

today = date.today()
today=pd.Timestamp(today).replace(tzinfo=dt.timezone.utc) 

In [112]:
data['created_at'] = pd.to_datetime(data['created_at'])

In [113]:
data['days_old_account'] = (today - data['created_at']).dt.days

In [114]:
data.to_parquet(
            f'../../data/02-randomize/{country}/02-variables/variables_batch2.parquet'
            )

In [147]:
data= pd.read_parquet(
            f'../../data/02-randomize/SA/02-variables/variables_batch2.parquet'
            )